## 🔑 Lab #4 과제 정답지

#### [문제 1] 정답

In [ ]:
import pandas as pd
from pycaret.regression import *

# 데이터 불러오기
path = '../../datasets/ml/bike-sharing/SeoulBikeData.csv'
df = pd.read_csv(path, encoding='cp949')
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

# 실험 환경 설정
reg_exp_lab = setup(
    data=df,
    target='RentedBikeCount',
    session_id=2024,
    train_size=0.8,
    normalize=True,
    transformation=True,
    date_features=['Date'],
    silent=True
)

#### [문제 2] 정답

In [ ]:
# 1. 상위 3개 모델 탐색
top3_models = compare_models(n_select=3)

# 2. 3개 모델 튜닝
tuned_top3 = []
for model in top3_models:
    tuned_model = tune_model(model, verbose=False)
    tuned_top3.append(tuned_model)

print("튜닝된 상위 3개 모델:", tuned_top3)

#### [문제 3] 정답

In [ ]:
# 1. 블렌딩 모델 생성
blended_model = blend_models(estimator_list=tuned_top3, verbose=False)

# 2. 스태킹 모델 생성
stacked_model = stack_models(estimator_list=tuned_top3[1:], meta_model=tuned_top3[0], verbose=False)

print("블렌딩 모델:", blended_model)
print("\n스태킹 모델:", stacked_model)

#### [문제 4] 정답

In [ ]:
# 성능을 기록할 딕셔너리
performance_summary = {}
all_models_to_check = tuned_top3 + [blended_model, stacked_model]

# 모델 이름 설정 (가독성을 위해)
model_names = [
    f'Tuned_{m.__class__.__name__}' for m in tuned_top3
] + ['Blended_Tuned_Top3', 'Stacked_Tuned_Top3']

# 각 모델의 성능 측정
for name, model in zip(model_names, all_models_to_check):
    # predict_model을 실행하고 결과에서 RMSE 값 추출
    rmse = predict_model(model, verbose=False).loc['RMSE', 'Value']
    performance_summary[name] = rmse

# DataFrame으로 변환 및 정렬하여 출력
summary_df = pd.DataFrame.from_dict(performance_summary, orient='index', columns=['RMSE'])
print("--- 모델별 최종 성능 (RMSE) ---")
print(summary_df.sort_values('RMSE'))

#### [문제 5] 정답

In [ ]:
# 위 결과에서 가장 RMSE가 낮은 모델을 선택
# 예를 들어, stacked_model이 가장 성능이 좋았다고 가정
best_performing_model = stacked_model 

# 1. 특성 중요도 시각화
print("\n--- 최고 성능 모델의 특성 중요도 ---")
plot_model(best_performing_model, plot='feature')

# 2. 모델 최종화 및 저장
final_best_model = finalize_model(best_performing_model)
save_model(final_best_model, 'my_best_bike_model')

print("\n✅ 최고 성능 모델이 'my_best_bike_model.pkl'로 저장되었습니다.")